In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import random
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel, BertForSequenceClassification
#from transformers import AdamW, 
from transformers import AutoTokenizer
from transformers import AutoModel
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support,f1_score
from collections import Counter

from datasets import load_dataset


from ml_moo.scalarization.two_objs_hate_speech import HateSpeechScalarization
from ml_moo import moo
from ml_moo.analysis.pareto_frontier import ParetoFrontier

/home/lineccsa/mestrado/moo_research/moo_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Definir seed para reprodutibilidade
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

In [4]:
# Carregar o dataset HateXplain
dataset = load_dataset("Hate-speech-CNERG/hatexplain", trust_remote_code=True)


In [5]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'annotators', 'rationales', 'post_tokens'],
        num_rows: 15383
    })
    validation: Dataset({
        features: ['id', 'annotators', 'rationales', 'post_tokens'],
        num_rows: 1922
    })
    test: Dataset({
        features: ['id', 'annotators', 'rationales', 'post_tokens'],
        num_rows: 1924
    })
})


In [6]:
# model_name = "tum-nlp/bert-hateXplain"
#"hate-bert, uncased bert"
model_name="bert-base-uncased"

In [7]:
print("GPU disponível:", torch.cuda.is_available())

GPU disponível: True


In [8]:
print("Número de GPUs:", torch.cuda.device_count())

Número de GPUs: 1


In [9]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

## Multi task model

In [10]:
class MultiTaskHateSpeechDataset(Dataset):
    def __init__(self, dataset, tokenizer_name="bert-base-uncased", max_length=128):
        self.dataset = dataset
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        self.max_length = max_length
        self.bert_model = AutoModel.from_pretrained(tokenizer_name)
        self.bert_model.eval()

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        data = self.dataset[idx]

        # Converter lista de tokens em texto
        text = " ".join(data["post_tokens"])

        # Encontrar a moda dos rótulos (classificação de discurso de ódio)
        labels = data["annotators"]["label"]
        label_counts = Counter(labels)
        majority_label = label_counts.most_common(1)[0][0]  # Pega a moda
      
        label_hate = 0 if majority_label == 1 else 1

        # Criar rótulo para discurso de ódio contra gênero (exemplo: "women")
        target_groups = data["annotators"]["target"]
        #label_target = 1 if len(target_groups) > 0 else 0
        hate_women = 0
        hate_homosexual = 0
        hate_indigenous = 0
        hate_african = 0
        hate_asian = 0
        hate_jewish = 0

        for i in range(len(target_groups)):
            #print(target_groups[i])
            if label_hate == 1:
                if ('Women' in target_groups[i]):
                    hate_women = 1
                if ('Homosexual' in target_groups[i]):
                    hate_homosexual = 1
                if ('Indigenous' in target_groups[i]):
                    hate_indigenous = 1
                if ('African' in target_groups[i]):
                    hate_african = 1
                if ('Asian' in target_groups[i]):
                    hate_asian = 1
                if ('Jewish' in target_groups):
                    hate_jewish = 1

        # Tokenizar texto
        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "label": torch.tensor(label_hate, dtype=torch.long),
            "women_hate": torch.tensor(hate_women, dtype=torch.long),
            "homosexual_hate": torch.tensor(hate_homosexual, dtype=torch.long),
            "indigenous_hate": torch.tensor(hate_indigenous, dtype=torch.long),
            "african_hate": torch.tensor(hate_african, dtype=torch.long),
            "asian_hate": torch.tensor(hate_asian, dtype=torch.long),
            "jewish_hate": torch.tensor(hate_jewish, dtype=torch.long),    
        }

In [11]:
# Criar os datasets para treino, validação e teste
train_dataset = MultiTaskHateSpeechDataset(dataset["train"])
val_dataset = MultiTaskHateSpeechDataset(dataset["train"])
test_dataset = MultiTaskHateSpeechDataset(dataset["train"])

# DataLoaders
batch_size = 8
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_dataloader = DataLoader(val_dataset, batch_size=batch_size)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

In [12]:
class MultiTaskModel(nn.Module):
    def __init__(self, model_name="tum-nlp/bert-hateXplain", num_labels=2, dropout_rate=0.3):
        super(MultiTaskModel, self).__init__()
        
        # Camada compartilhada: BERT
        self.shared_bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.shared_bert.config.hidden_size
        # print("hidden size", hidden_size)
        
        # Camadas adicionais compartilhadas
        self.dropout = nn.Dropout(dropout_rate)
        self.shared_fc = nn.Linear(hidden_size, 128)

        self.classifier_hate = nn.Linear(128, num_labels)  
        self.classifier_women = nn.Linear(128, num_labels)
        self.classifier_homosexual = nn.Linear(128, num_labels)
        self.classifier_indigenous = nn.Linear(128, num_labels)
        self.classifier_african = nn.Linear(128, num_labels)
        self.classifier_asian = nn.Linear(128, num_labels)
        self.classifier_jewish = nn.Linear(128, num_labels)

    def forward(self, input_ids, attention_mask):
        # Passagem pela parte compartilhada
        outputs = self.shared_bert(input_ids=input_ids, attention_mask=attention_mask)

        pooled_output = outputs.last_hidden_state[:, 0, :]  # Token [CLS]
        pooled_output = self.dropout(pooled_output)
        
        hate_features = self.shared_fc(pooled_output)
        women_features = self.shared_fc(pooled_output)
        homosexual_features = self.shared_fc(pooled_output)
        indigenous_features = self.shared_fc(pooled_output)
        african_features = self.shared_fc(pooled_output)
        asian_features = self.shared_fc(pooled_output)
        jewish_features = self.shared_fc(pooled_output)
        
        # Classificação final
        logits_hate = self.classifier_hate(hate_features)
        logits_women = self.classifier_women(women_features)
        logits_homosexual = self.classifier_homosexual(homosexual_features)
        logits_indigenous = self.classifier_indigenous(indigenous_features)
        logits_african = self.classifier_african(african_features)
        logits_asian = self.classifier_asian(asian_features)
        logits_jewish = self.classifier_jewish(jewish_features)

        return logits_hate, logits_women, logits_homosexual, logits_indigenous, logits_african, logits_asian, logits_jewish

In [13]:
def train_model(model, train_loader, val_loader, epochs=3):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = MultiTaskModel().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss()

    best_val_f1 = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} - Training"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels_hate = batch["label"].to(device)
            labels_women = batch["women_hate"].to(device)
            labels_homosexual = batch["homosexual_hate"].to(device)
            labels_indigenous = batch["indigenous_hate"].to(device)
            labels_african = batch["african_hate"].to(device)
            labels_asian = batch["asian_hate"].to(device)
            labels_jewish = batch["jewish_hate"].to(device)
            
            optimizer.zero_grad()
                
            logits_hate, logits_women, logits_homosexual, \
            logits_indigenous, logits_african, logits_asian, \
            logits_jewish = model(input_ids, attention_mask)

            # Calcular perdas
            loss_hate = criterion(logits_hate, labels_hate)
            loss_women = criterion(logits_women, labels_women)
            loss_homosexual = criterion(logits_homosexual, labels_homosexual)
            loss_indigenous = criterion(logits_indigenous, labels_indigenous)
            loss_african = criterion(logits_african, labels_african)
            loss_asian = criterion(logits_asian, labels_asian)
            loss_jewish = criterion(logits_jewish, labels_jewish)
            
            # Balancear as perdas
            loss = (loss_hate + loss_women + loss_homosexual + loss_indigenous + loss_african + loss_asian + loss_jewish) / 7
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
        avg_train_loss = train_loss / len(train_loader)
        print(f"Epoch {epoch+1} - Average training loss: {avg_train_loss:.4f}")

        # Validação
        val_metrics = evaluate_model(model, val_loader, device)
        print(f"Validation metrics - Hate F1: {val_metrics['hate_f1']:.4f}, women F1: {val_metrics['women_f1']:.4f}, homosexual F1: {val_metrics['homosexual_f1']:.4f}")
        print(f"Validation metrics - indigenous F1: {val_metrics['indigenous_f1']:.4f}, african F1: {val_metrics['african_f1']:.4f}, asian F1: {val_metrics['asian_f1']:.4f}, jewish F1: {val_metrics['jewish_f1']:.4f}")
        print(f"Validation metrics - Hate ACC: {val_metrics['hate_acc']:.4f}, women ACC: {val_metrics['women_acc']:.4f}, homosexual ACC: {val_metrics['homosexual_acc']:.4f}")
        print(f"Validation metrics - indigenous ACC: {val_metrics['indigenous_acc']:.4f}, african ACC: {val_metrics['african_acc']:.4f}, asian ACC: {val_metrics['asian_acc']:.4f}, jewish ACC: {val_metrics['jewish_acc']:.4f}")
        

    return model

In [14]:
# Função de avaliação
def evaluate_model(model, dataloader, device):
    model.eval()
    
    all_preds_hate = []
    all_true_hate = []

    all_preds_women = []
    all_true_women = []

    all_preds_homosexual = []
    all_true_homosexual = []

    all_preds_indigenous = []
    all_true_indigenous = []

    all_preds_african = []
    all_true_african = []

    all_preds_asian = []
    all_true_asian = []

    all_preds_jewish = []
    all_true_jewish = []

    total_loss_hate = 0.0
    total_loss_women = 0.0
    total_loss_homosexual = 0.0
    total_loss_indigenous = 0.0
    total_loss_african = 0.0
    total_loss_asian = 0.0
    total_loss_jewish = 0.0
    total_batches = 0
    
    # Definir os critérios de loss (provavelmente CrossEntropyLoss para classificação)
    loss_fn = torch.nn.CrossEntropyLoss()
 
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels_hate = batch["label"].to(device)
            labels_women = batch["women_hate"].to(device)
            labels_homosexual = batch["homosexual_hate"].to(device)
            labels_indigenous = batch["indigenous_hate"].to(device)
            labels_african = batch["african_hate"].to(device)
            labels_asian = batch["asian_hate"].to(device)
            labels_jewish = batch["jewish_hate"].to(device)
            
            logits_hate, logits_women, logits_homosexual, \
            logits_indigenous, logits_african, logits_asian, \
            logits_jewish = model(input_ids, attention_mask)
            
            # Calcular loss para cada task
            loss_hate = loss_fn(logits_hate, labels_hate)
            loss_women = loss_fn(logits_women, labels_women)
            loss_homosexual = loss_fn(logits_homosexual, labels_homosexual)
            loss_indigenous = loss_fn(logits_indigenous, labels_indigenous)
            loss_african = loss_fn(logits_african, labels_african)
            loss_asian = loss_fn(logits_asian, labels_asian)
            loss_jewish = loss_fn(logits_jewish, labels_jewish)
            
            # Acumular losses
            total_loss_hate += loss_hate.item()
            total_loss_women += loss_women.item()
            total_loss_homosexual += loss_homosexual.item()
            total_loss_indigenous += loss_indigenous.item()
            total_loss_african += loss_african.item()
            total_loss_asian += loss_asian.item()
            total_loss_jewish += loss_jewish.item()
            total_batches += 1
            
            preds_hate = torch.argmax(logits_hate, dim=1).cpu().numpy()
            preds_women = torch.argmax(logits_women, dim=1).cpu().numpy()
            preds_homosexual = torch.argmax(logits_homosexual, dim=1).cpu().numpy()
            preds_indigenous = torch.argmax(logits_indigenous, dim=1).cpu().numpy()
            preds_african = torch.argmax(logits_african, dim=1).cpu().numpy()
            preds_asian = torch.argmax(logits_asian, dim=1).cpu().numpy()
            preds_jewish = torch.argmax(logits_jewish, dim=1).cpu().numpy()
            
            all_preds_hate.extend(preds_hate)
            all_true_hate.extend(labels_hate.cpu().numpy())

            all_preds_women.extend(preds_women)
            all_true_women.extend(labels_women.cpu().numpy())

            all_preds_homosexual.extend(preds_homosexual)
            all_true_homosexual.extend(labels_homosexual.cpu().numpy())
    
            all_preds_indigenous.extend(preds_indigenous)
            all_true_indigenous.extend(labels_indigenous.cpu().numpy())

            all_preds_african.extend(preds_african)
            all_true_african.extend(labels_african.cpu().numpy())
    
            all_preds_asian.extend(preds_asian)
            all_true_asian.extend(labels_asian.cpu().numpy())

            all_preds_jewish.extend(preds_jewish)
            all_true_jewish.extend(labels_jewish.cpu().numpy())

     # Calcular médias das losses
    avg_loss_hate = total_loss_hate / total_batches
    avg_loss_women = total_loss_women / total_batches
    avg_loss_homosexual = total_loss_homosexual / total_batches
    avg_loss_indigenous = total_loss_indigenous / total_batches
    avg_loss_african = total_loss_african / total_batches
    avg_loss_asian = total_loss_asian / total_batches
    avg_loss_jewish = total_loss_jewish / total_batches
    
    # Métricas
    hate_acc = accuracy_score(all_true_hate, all_preds_hate)
    hate_f1 = f1_score(all_true_hate, all_preds_hate)

    women_acc = accuracy_score(all_true_women, all_preds_women)
    women_f1 = f1_score(all_true_women, all_preds_women)

    homosexual_acc = accuracy_score(all_true_homosexual, all_preds_homosexual)
    homosexual_f1 = f1_score(all_true_homosexual, all_preds_homosexual)
    
    indigenous_acc = accuracy_score(all_true_indigenous, all_preds_indigenous)
    indigenous_f1 = f1_score(all_true_indigenous, all_preds_indigenous)

    african_acc = accuracy_score(all_true_african, all_preds_african)
    african_f1 = f1_score(all_true_african, all_preds_african)

    asian_acc = accuracy_score(all_true_asian, all_preds_asian)
    asian_f1 = f1_score(all_true_asian, all_preds_asian)

    jewish_acc = accuracy_score(all_true_jewish, all_preds_jewish)
    jewish_f1 = f1_score(all_true_jewish, all_preds_jewish)

    return {
        "hate_acc": hate_acc,
        "hate_f1": hate_f1,
        "hate_loss": avg_loss_hate,
        "women_acc": women_acc,
        "women_f1": women_f1,
        "women_loss": avg_loss_women,
        "homosexual_acc": homosexual_acc,
        "homosexual_f1": homosexual_f1,
        "homosexual_loss": avg_loss_homosexual,
        "indigenous_acc": indigenous_acc,
        "indigenous_f1": indigenous_f1,
        "indigenous_loss": avg_loss_indigenous,
        "african_acc": african_acc,
        "african_f1": african_f1,
        "african_loss": avg_loss_african,
        "asian_acc": asian_acc,
        "asian_f1": asian_f1,
        "asian_loss": avg_loss_asian,
        "jewish_acc": jewish_acc,
        "jewish_f1": jewish_f1,
        "jewish_loss": avg_loss_jewish,
    }

In [17]:
# Instanciar e treinar o modelo
model = MultiTaskModel(model_name)
model = train_model(model, train_dataloader, valid_dataloader, epochs=3)

# Avaliar no conjunto de teste
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_metrics = evaluate_model(model, test_dataloader, device)

print("\nTest Results:")
print(f"Hate Speech Detection - Accuracy: {test_metrics['hate_acc']:.4f}, F1: {test_metrics['hate_f1']:.4f}, Loss: {test_metrics['hate_loss']:.4f}")
print(f"women Classification - Accuracy: {test_metrics['women_acc']:.4f}, F1: {test_metrics['women_f1']:.4f}, Loss: {test_metrics['women_loss']:.4f}")
print(f"homosexual - Accuracy: {test_metrics['homosexual_acc']:.4f}, F1: {test_metrics['homosexual_f1']:.4f}, Loss: {test_metrics['homosexual_loss']:.4f}")
print(f"indigenous Classification - Accuracy: {test_metrics['indigenous_acc']:.4f}, F1: {test_metrics['indigenous_f1']:.4f}, Loss: {test_metrics['indigenous_loss']:.4f}")
print(f"african - Accuracy: {test_metrics['african_acc']:.4f}, F1: {test_metrics['african_f1']:.4f}, Loss: {test_metrics['african_loss']:.4f}")
print(f"asian Classification - Accuracy: {test_metrics['asian_acc']:.4f}, F1: {test_metrics['asian_f1']:.4f}, Loss: {test_metrics['asian_loss']:.4f}")
print(f"jewish - Accuracy: {test_metrics['jewish_acc']:.4f}, F1: {test_metrics['jewish_f1']:.4f}, Loss: {test_metrics['jewish_loss']:.4f}")


Epoch 1/3 - Training: 100%|██████████| 1923/1923 [00:30<00:00, 63.08it/s]


Epoch 1 - Average training loss: 0.2236


Evaluating: 100%|██████████| 1923/1923 [00:11<00:00, 162.15it/s]
/home/lineccsa/mestrado/moo_research/moo_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Validation metrics - Hate F1: 0.8476, women F1: 0.4118, homosexual F1: 0.7638
Validation metrics - indigenous F1: 0.0000, african F1: 0.8101, asian F1: 0.0150, jewish F1: 0.0000
Validation metrics - Hate ACC: 0.8095, women ACC: 0.8845, homosexual ACC: 0.9544
Validation metrics - indigenous ACC: 0.9966, african ACC: 0.9265, asian ACC: 0.9744, jewish ACC: 1.0000


Epoch 2/3 - Training: 100%|██████████| 1923/1923 [00:32<00:00, 59.35it/s]


...

Epoch 3 - Average training loss: 0.1504


Evaluating: 100%|██████████| 1923/1923 [00:12<00:00, 151.67it/s]
/home/lineccsa/mestrado/moo_research/moo_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Validation metrics - Hate F1: 0.8977, women F1: 0.5822, homosexual F1: 0.8524
Validation metrics - indigenous F1: 0.0000, african F1: 0.8641, asian F1: 0.6685, jewish F1: 0.0000
Validation metrics - Hate ACC: 0.8760, women ACC: 0.9095, homosexual ACC: 0.9717
Validation metrics - indigenous ACC: 0.9966, african ACC: 0.9487, asian ACC: 0.9847, jewish ACC: 1.0000


Evaluating: 100%|██████████| 1923/1923 [00:13<00:00, 138.74it/s]
/home/lineccsa/mestrado/moo_research/moo_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Test Results:
Hate Speech Detection - Accuracy: 0.8760, F1: 0.8977, Loss: 0.3028
women Classification - Accuracy: 0.9095, F1: 0.5822, Loss: 0.2351
homosexual - Accuracy: 0.9717, F1: 0.8524, Loss: 0.0820
indigenous Classification - Accuracy: 0.9966, F1: 0.0000, Loss: 0.0221
african - Accuracy: 0.9487, F1: 0.8641, Loss: 0.1369
asian Classification - Accuracy: 0.9847, F1: 0.6685, Loss: 0.0492
jewish - Accuracy: 1.0000, F1: 0.0000, Loss: 0.0004
